In [ ]:
#Install needed libraries and dependencies
!pip uninstall -y vllm transformers tokenizers
!pip install vllm==0.6.4.post1 transformers==4.46.3 tokenizers==0.20.3 accelerate pandas

Sarvam-1

In [ ]:
#Tier 3 Voting
import os
import gc
import torch
import pandas as pd
from collections import Counter
from transformers import AutoTokenizer, AutoModelForCausalLM
from google.colab import drive
from tqdm.notebook import tqdm

gc.collect()
torch.cuda.empty_cache()
drive.mount('/content/drive')

MODEL_NAME = "sarvamai/sarvam-1"
MASTER_CSV_PATH = "PATH_OF_TIER_2_OUTPUT_RATIONALES"
OUTPUT_CSV_PATH = "PATH_OF_OUTPUT"

device = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Loading {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto"
)

print(f"Loading Master Rationale Data...")
df = pd.read_csv(MASTER_CSV_PATH)

target_map = {'option1': 'A', 'option2': 'B', 'option3': 'C', 'option4': 'D'}
df['True_Label'] = df['target'].map(target_map)

def get_best_letter_from_logits(prompt_text):
    """
    Extracts the highest probability letter (A, B, C, or D)
    from the model's next-token logits.
    """
    inputs = tokenizer(prompt_text, return_tensors="pt").to(device)

    with torch.no_grad():
        outputs = model(**inputs)
        next_token_logits = outputs.logits[0, -1, :]

    choices = ['A', 'B', 'C', 'D']
    choice_ids = [tokenizer.encode(c, add_special_tokens=False)[-1] for c in choices]

    filtered_logits = next_token_logits[choice_ids]
    best_index = torch.argmax(filtered_logits).item()

    return choices[best_index]

print("\nRunning Inference and Calculating Majority Votes...")
results_log = []
correct_count = 0

for i, row in tqdm(df.iterrows(), total=len(df), desc="Scoring Progress"):
    options_block = f"A. {row['option1']}\nB. {row['option2']}\nC. {row['option3']}\nD. {row['option4']}"

    rationales = [
        row.get('rationale_Logical', "No rationale provided."),
        row.get('rationale_Balanced', "No rationale provided."),
        row.get('rationale_Creative', "No rationale provided.")
    ]

    preds = []
    for rationale in rationales:
        prompt = (
            f"### RATIONALE\n{rationale}\n\n"
            f"### QUESTION\n{row['question']}\n\n"
            f"### OPTIONS\n{options_block}\n\n"
            f"The correct option is:"
        )

        pred = get_best_letter_from_logits(prompt)
        preds.append(pred)

    pred_logical, pred_balanced, pred_creative = preds

    counts = Counter(preds)
    max_count = max(counts.values())
    tied_options = [opt for opt, count in counts.items() if count == max_count]

    if len(tied_options) == 1:
        final_pred = tied_options[0]
    else:
#Tiebreaker
        final_pred = pred_logical

    truth = row['True_Label']
    is_correct = (final_pred == truth)
    if is_correct:
        correct_count += 1

    results_log.append({
        "Question_ID": i,
        "True_Label": truth,
        "Pred_Logical": pred_logical,
        "Pred_Balanced": pred_balanced,
        "Pred_Creative": pred_creative,
        "Final_Voted_Pred": final_pred,
        "Is_Correct": is_correct
    })

accuracy = (correct_count / len(df)) * 100
print(f"\nENSEMBLE VOTING COMPLETE!")
print(f"FINAL ACCURACY: {accuracy:.2f}%")

final_df = pd.DataFrame(results_log)
final_df.to_csv(OUTPUT_CSV_PATH, index=False)
print(f"Voting breakdown saved to:\n{OUTPUT_CSV_PATH}")